## Splitting full articles into snippets

### What we are doing
We have 2,700 full newspaper articles that we want to classify using the three focus categories identified above (Actor categories, Character Role, Conflict types). To do this, we need to break each article into snippets that can be passed to the classifier one at a time.

We use a **sentence-grouped sliding window** approach:
- Sentences are grouped together until a target word count is reached (60–80 words)
- The window then advances by one sentence at a time (one-sentence overlap between consecutive windows)
- Snippets shorter than 20 words are discarded as uninformative fragments

### Why this approach
Our labeled training data consists of snippets manually selected by human coders. The structural analysis above shows that the vast majority of these fall between 30 and 100 words, with a median of 53 words. The sentence-grouped sliding window is the closest mechanical approximation to this process: it preserves sentence boundaries (so no clause is cut mid-way), and the one-sentence overlap ensures that content spanning two consecutive windows is not missed.

Fixed-word windows were rejected because they cut sentences arbitrarily, degrading the linguistic cues (actor mentions, role attributions, conflict framings) that the classifier relies on. Paragraph-based splits were rejected because newspaper paragraphs vary too widely in length and often fall outside the 30–100 word range of the training data.

### What to expect
Most snippets produced by this mechanical split will not contain codeable content — they cover background information, transitions, or context unrelated to the conflict framing categories. This is normal. A relevance filter is applied to the sliding-window output to reduce noise: only snippets scoring ≥ 2 on a keyword-based relevance heuristic are retained and saved as `snippets_qual_filter.csv`, which is the input to `NB3A_Conflict_Codes_Training.ipynb`.


In [31]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

LN_PATH = '../ln_preprocessing/ln_data_final.csv'

ln_full = pd.read_csv(LN_PATH)
print(f"Total articles in ln_data_final   : {ln_full['ID'].nunique()}")

fname = '../ln_labeling/labeled_data_n154.xlsx'
training = pd.read_excel(fname)
training = training[['Text', 'Source', 'Codes', 'Number of Codes']]

# ── get training article IDs ─────────────────────────────────────────────────
# training 'Source' column has a .txt suffix: strip it to match
training_ids = set(
    training['Source'].dropna()
    .str.replace(r'\.txt$', '', regex=True)
    .unique()
)
print(f"Unique articles in training set   : {len(training_ids)}")

# ── exclude training articles ────────────────────────────────────────────────
ln_inference = ln_full[~ln_full['ID'].isin(training_ids)].reset_index(drop=True)

print(f"Articles excluded (in training)   : {len(ln_full) - len(ln_inference)}")
print(f"Distinct IDs remaining            : {ln_inference['ID'].nunique()}")
ln_inference.head(3)

# ── outlet-level quality filter ─────────────────────────────────────────────
# Outlet_Name format: 'Outlet Name, 1234words' — match on startswith
DROP_OUTLETS = {
    'CE Noticias Financieras English',
    'Metal Bulletin Daily Alerts',
    'American Metal Market (AMM)',
    'Commodity Online',
    'Proactive Investors (UK)',
    'MENAFN - Market Reports (English)',
}

mask_drop = ln_inference['Outlet_Name'].fillna('').apply(
    lambda x: any(x.startswith(o) for o in DROP_OUTLETS)
)
ln_inference = ln_inference[~mask_drop].reset_index(drop=True)
print(f"Articles dropped (low-quality outlets): {mask_drop.sum()}")
print(f"Articles remaining for inference       : {len(ln_inference)}")


Total articles in ln_data_final   : 2334
Unique articles in training set   : 151
Articles excluded (in training)   : 151
Distinct IDs remaining            : 2183
Articles dropped (low-quality outlets): 457
Articles remaining for inference       : 1726


In [32]:
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

TARGET_MIN  = 60   # start a new window once we hit this
TARGET_MAX  = 80   # hard cap — if a single sentence exceeds this we keep it anyway
MIN_SNIPPET = 20   # discard fragments shorter than this
OVERLAP     = 1    # sentences carried over into the next window

def sliding_window_snippets(text, target_min=TARGET_MIN, target_max=TARGET_MAX,
                             min_words=MIN_SNIPPET, overlap=OVERLAP):
    sentences = sent_tokenize(str(text))
    snippets = []
    i = 0
    while i < len(sentences):
        window = []
        word_count = 0
        j = i
        while j < len(sentences):
            w = len(sentences[j].split())
            if word_count + w > target_max and window:
                break
            window.append(sentences[j])
            word_count += w
            j += 1
            if word_count >= target_min:
                break
        snippet = ' '.join(window).strip()
        if len(snippet.split()) >= min_words:
            snippets.append(snippet)
        # advance by (window size - overlap), minimum 1
        advance = max(1, len(window) - overlap)
        i += advance
    return snippets

# ── apply to all inference articles ─────────────────────────────────────────
rows = []
for _, article in ln_inference.iterrows():
    for snippet in sliding_window_snippets(article['Text_body']):
        rows.append({
            'article_id'  : article['ID'],
            'title'       : article['Title'],
            'outlet'      : article['Outlet_Name'],
            'date'        : article['Date'],
            'snippet'     : snippet,
            'word_count'  : len(snippet.split()),
        })

snippets_df = pd.DataFrame(rows)

# ── summary ──────────────────────────────────────────────────────────────────
print(f"Articles processed          : {ln_inference['ID'].nunique()}")
print(f"Total snippets generated    : {len(snippets_df)}")
print(f"Avg snippets per article    : {len(snippets_df) / ln_inference['ID'].nunique():.1f}")
print(f"\nSnippet word count:")
print(snippets_df['word_count'].describe().round(1))
print(f"\nSnippets < {MIN_SNIPPET} words (discarded): already excluded")
print(f"Snippets 20–60 words  : {((snippets_df['word_count'] >= 20) & (snippets_df['word_count'] < 60)).sum()}")
print(f"Snippets 60–80 words  : {((snippets_df['word_count'] >= 60) & (snippets_df['word_count'] <= 80)).sum()}")
print(f"Snippets > 80 words   : {(snippets_df['word_count'] > 80).sum()}")

snippets_df.head(3)

Articles processed          : 1726
Total snippets generated    : 73485
Avg snippets per article    : 42.6

Snippet word count:
count    73485.0
mean        65.6
std         31.3
min         20.0
25%         59.0
50%         65.0
75%         72.0
max       2087.0
Name: word_count, dtype: float64

Snippets < 20 words (discarded): already excluded
Snippets 20–60 words  : 18983
Snippets 60–80 words  : 52932
Snippets > 80 words   : 1570


,article_id,title,outlet,date,snippet,word_count
0,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,Mining holds the key to a green future - no wo...,68
1,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,"Not any more. Today, the shallow sandbank, loc...",62
2,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,Welcome to the beginning of the end of the fos...,73


## Snippet split — sanity check

**What to expect:** A typical LexisNexis article in this dataset runs ~500–900 words. With 60–80 word windows and one-sentence overlap, each article should yield roughly 8–15 snippets, putting the total somewhere between 15,000 and 30,000 snippets.

**Distribution check:** The median snippet word count should sit in the 60–80 word range. A long tail above 80 words is normal — it reflects articles with long sentences that exceed the target cap but are kept intact. A secondary bump below 60 words is also expected for article-ending fragments (last sentence(s) of an article that don't fill a full window but pass the 20-word minimum).

**Red flags to watch for:**
- Median well below 50 words → windows are too short; consider raising `TARGET_MIN`
- Very high snippet count (>50 per article on average) → articles may contain boilerplate/repeated text that inflates the count
- Many snippets > 150 words → articles likely contain very long run-on sentences; consider splitting on punctuation as a fallback

In [33]:
print(f"Articles processed         : {ln_inference['ID'].nunique()}")
print(f"Total snippets             : {len(snippets_df)}")
print(f"Avg snippets per article   : {len(snippets_df) / ln_inference['ID'].nunique():.1f}")
print(f"\nWord count distribution:")
print(snippets_df['word_count'].describe().round(1))
print(f"\nSnippets 20–59 words  : {((snippets_df['word_count'] >= 20) & (snippets_df['word_count'] < 60)).sum()}")
print(f"Snippets 60–80 words  : {((snippets_df['word_count'] >= 60) & (snippets_df['word_count'] <= 80)).sum()}")
print(f"Snippets 81–150 words : {((snippets_df['word_count'] > 80) & (snippets_df['word_count'] <= 150)).sum()}")
print(f"Snippets > 150 words  : {(snippets_df['word_count'] > 150).sum()}")

Articles processed         : 1726
Total snippets             : 73485
Avg snippets per article   : 42.6

Word count distribution:
count    73485.0
mean        65.6
std         31.3
min         20.0
25%         59.0
50%         65.0
75%         72.0
max       2087.0
Name: word_count, dtype: float64

Snippets 20–59 words  : 18983
Snippets 60–80 words  : 52932
Snippets 81–150 words : 1036
Snippets > 150 words  : 534


## Snippet split — results

**6 low-quality outlets dropped** (CE Noticias Financieras, Metal Bulletin Daily Alerts, American Metal Market, Commodity Online, Proactive Investors UK, MENAFN Market Reports), removing 457 articles. This reduces the inference set from 2,181 to **1,724 articles**.

The word count distribution is healthy: median 65 words, and the majority of snippets land in the 60–80 word target band.

**Long tail:** a small number of snippets exceed 150 words. These are single sentences too long to split further — likely tables, bullet lists run together, or garbled OCR. Small in number but worth filtering before classification.

In [34]:
# top 5 articles by snippet count
top5_ids = (
    snippets_df.groupby('article_id')
    .size()
    .sort_values(ascending=False)
    .head(5)
    .reset_index()
    .rename(columns={0: 'snippet_count'})
)

# join with full article text for manual inspection
top5 = top5_ids.merge(ln_inference[['ID', 'Title', 'Outlet_Name', 'Date', 'Text_body']],
                      left_on='article_id', right_on='ID').drop(columns='ID')


In [35]:
# ── keyword presence per snippet ─────────────────────────────────────────────
keywords = ['china', 'indonesia', 'nickel']

text_lower = snippets_df['snippet'].str.lower()
for kw in keywords:
    snippets_df[kw] = text_lower.str.contains(kw, regex=False).astype(int)

total = len(snippets_df)

# ── co-occurrence matrix ──────────────────────────────────────────────────────
cooc = pd.DataFrame(index=keywords, columns=keywords, dtype=int)
for a in keywords:
    for b in keywords:
        cooc.loc[a, b] = (snippets_df[a] & snippets_df[b]).sum()

cooc = cooc.fillna(0).astype(int)  # ensure integer values, no float in the heatmap input

# individual counts on diagonal are already correct (kw & kw = kw)
print("Co-occurrence counts (snippets containing both):")
print(cooc)
print(f"\nTotal snippets: {total}")
for kw in keywords:
    print(f"  '{kw}' present: {snippets_df[kw].sum()} ({snippets_df[kw].mean()*100:.1f}%)")
    

Co-occurrence counts (snippets containing both):
           china  indonesia  nickel
china      12175       4594    2556
indonesia   4594      15804    5204
nickel      2556       5204   11122

Total snippets: 73485
  'china' present: 12175 (16.6%)
  'indonesia' present: 15804 (21.5%)
  'nickel' present: 11122 (15.1%)


In [36]:
scmp = snippets_df[snippets_df['outlet'].str.contains('South China Morning Post', case=False, na=False)].copy()
print(f'SCMP snippets     : {len(scmp)}')
print(f'Distinct articles : {scmp["article_id"].nunique()}')

OUT = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_tests\snippets_scmp.xlsx'
scmp.to_excel(OUT, index=False)
print(f'Saved: {OUT}')

SCMP snippets     : 1559
Distinct articles : 62
Saved: C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_tests\snippets_scmp.xlsx


In [37]:
asia_times = snippets_df[snippets_df['outlet'].str.contains('Asia Times', case=False, na=False)].copy()
print(f'Asia Times snippets : {len(asia_times)}')
print(f'Distinct articles   : {asia_times["article_id"].nunique()}')

OUT = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_tests\snippets_asia_times.xlsx'
asia_times.to_excel(OUT, index=False)
print(f'Saved: {OUT}')


Asia Times snippets : 867
Distinct articles   : 26
Saved: C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_tests\snippets_asia_times.xlsx


In [38]:
OUT = r'C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_df.xlsx'
snippets_df.to_excel(OUT, index=False)
print(f'Saved {len(snippets_df):,} snippets to: {OUT}')


Saved 73,485 snippets to: C:\Users\charlott\Dropbox (Personal)\Paper_Indonesia_Nickel\ln_training\snippets_df.xlsx


In [39]:
import re

def _compile(*terms):
    """Compile an OR-regex from terms matched as whole words (case-insensitive)."""
    alts = [r'\b' + re.escape(t.lower()) + r'\b' for t in terms]
    return re.compile('|'.join(alts), re.IGNORECASE)

def _hit(pat, text):
    return bool(pat.search(text))

# ── Anchor groups (Rules A–F) ─────────────────────────────────────────────────
A_LOC = _compile(
    'indonesia', 'indonesian', 'jakarta', 'sulawesi', 'central sulawesi',
    'southeast sulawesi', 'north maluku', 'morowali', 'weda bay', 'halmahera',
    'obi island', 'konawe', 'bahodopi', 'pomalaa', 'raja ampat',
    'west kalimantan', 'north kalimantan',
    'jokowi', 'joko widodo', 'prabowo', 'prabowo subianto', 'luhut',
    'luhut binsar panjaitan', 'muhammad lutfi', 'esdm', 'bkpm', 'ina',
    'bahlil', 'anies', 'gibran',
)
A_NICKEL = _compile(
    'nickel', 'smelter', 'smelting', 'ferronickel', 'npi', 'hpal', 'mhp',
    'battery material', 'battery materials', 'ev battery', 'electric vehicle battery',
    'battery supply chain', 'stainless steel', 'downstreaming', 'export ban',
    'industrial park', 'captive coal', 'nickel export ban', 'ore export ban',
    'resource nationalism', 'tailings pipeline', 'conflict-free nickel',
    'processing plant', 'nickel processing', 'nickel mining', 'nickel ore',
    'battery-materials boom', 'stainless steel boom', 'deep-sea tailings',
    'hpal plant', 'mhp plant',
)
B_SINGLE = _compile(
    'imip', 'iwip', 'pt gni', 'pt vdni', 'pt oss', 'morowali industrial park',
    'weda bay industrial park', 'antam', 'harita nickel', 'vale indonesia',
    'ibc', 'itss', 'pt vdn', 'pt virtue dragon', 'fscrir',
)
B_COMPOUND = [
    (_compile('huayou'),                       _compile('morowali')),
    (_compile('tsingshan'),                    _compile('morowali')),
    (_compile('norilsk nickel'),               _compile('indonesia')),
    (_compile('metallurgical corp of china'),  _compile('indonesia')),
]
def rule_b(t):
    if _hit(B_SINGLE, t): return True
    return any(_hit(a, t) and _hit(b, t) for a, b in B_COMPOUND)

C_CHINESE = _compile(
    'chinese dfi', 'chinese bank', 'chinese firm', 'chinese firms', 'chinese soe',
    'chinese government', 'chinese worker', 'chinese workers', 'chinese-backed',
    'chinese-owned', 'chinese-funded', 'chinese investor', 'chinese company',
    'chinese ambassador', 'chinese national', 'chinese nationals',
    'chinese survey vessel', 'chinese coast guard', 'chinese state councillor',
    'wang yi', 'chinese subcontractor', 'rptka', 'chinese batch arrival',
    'mainland worker', 'mainland workers', 'chinese labour', 'chinese labor',
    'chinese employees', 'chinese personnel', 'chinese migrant',
)
D_LOCAL = _compile(
    'resident', 'residents', 'villager', 'villagers', 'fishermen', 'fisherman',
    'indigenous', 'adat', 'local worker', 'local workers', 'local community',
    'walhi', 'jatam', 'celios', 'icel', 'greenpeace indonesia', 'trend asia',
    'aman', 'aeer', 'pwyp indonesia', 'union', 'unions', 'labour union',
    'labor union', 'trade union',
)
D_CONTEXT = _compile(
    'nickel', 'smelter', 'smelting', 'mining', 'morowali', 'weda bay',
    'sulawesi', 'halmahera', 'industrial park', 'imip', 'pt gni', 'pt vdni',
    'pt oss', 'antam', 'harita nickel', 'pollution', 'accident', 'explosion',
    'protest', 'strike', 'land dispute', 'labour dispute', 'labor dispute',
    'workplace safety', 'tailings', 'deforestation', 'health risk',
    'anti-chinese', 'deport workers', 'stealing jobs', 'resentment',
    'chinese workers', 'deportation',
)
E_INTL = _compile(
    'eu trade commissioner', 'valdis dombrovskis', 'wto panel', 'gatt',
    'us trade representative', 'ustr', 'wto appellate body',
    'international partners group', 'jetp', 'world bank', 'iea', 'g7',
    'glencore', 'rio tinto', 'bhp', 'volkswagen', 'hyundai', 'ford',
    'lg energy', 'samsung sdi', 'panasonic', 'posco', 'tesla',
    'global witness', 'amnesty international', 'survival international',
    'human rights watch', 'global voices',
    'european union', 'oecd', 'us department of labor', 'us state department',
    'asian development bank',
)
F_CHINA = _compile('china', 'chinese')
F_INDO  = _compile('indonesia', 'indonesian')
F_DIPLO = _compile(
    'indonesia-china', 'indonesia–china', 'bilateral cooperation',
    'bilateral investment', 'natuna', 'south china sea',
    'defence cooperation', 'defense cooperation',
    'indonesian defence minister', 'indonesian defense minister',
)
F_LINK = _compile(
    'nickel', 'mining', 'investment', 'mineral', 'battery', 'energy',
    'resources', 'eez', 'industrial park', 'smelter', 'downstreaming',
    'supply chain', 'critical mineral', 'battery material', 'infrastructure',
)

# ── Relevance score ───────────────────────────────────────────────────────────
# Weight:  4 = strong case-specific anchor (Rule B)
#          2 = specific domain / actor signal
#          1 = supporting / contextual signal
# C_CONTEXT and E_CONTEXT excluded from scoring: both contain 'indonesia',
# which would inflate scores and conflate any Indonesia snippet with relevance.
# Threshold >= 2 selected from calibration on 390 labeled snippets:
#   recall 83% overall, 74–93% per category (vs 51% / 30–67% for hard AND filter).

def relevance_score(text):
    t = str(text).lower()
    s = 0
    if rule_b(t):               s += 4   # strong case-specific anchor
    s += 2 * _hit(A_NICKEL, t)            # nickel/industrial domain
    s += 2 * _hit(C_CHINESE, t)           # Chinese actor
    s += 2 * _hit(D_LOCAL, t)             # local civil society
    s += 2 * _hit(E_INTL, t)              # international actor
    s += 2 * _hit(F_DIPLO, t)             # bilateral/diplomatic signal
    s += 1 * _hit(A_LOC, t)               # Indonesia/location (supporting)
    s += 1 * _hit(D_CONTEXT, t)           # conflict/project context (incl. 'mining','protest')
    s += 1 * int(_hit(F_CHINA, t) and _hit(F_INDO, t))  # China + Indonesia pair
    s += 1 * _hit(F_LINK, t)              # economic/resource link
    return s

THRESHOLD = 2

snippets_df['relevance_score'] = snippets_df['snippet'].apply(relevance_score)
snippets_df['pass_filter']     = snippets_df['relevance_score'] >= THRESHOLD

# ── Summary ───────────────────────────────────────────────────────────────────
n_total = len(snippets_df)
n_pass  = snippets_df['pass_filter'].sum()
print(f'Total snippets          : {n_total:,}')
print(f'Pass (score >= {THRESHOLD})       : {n_pass:,}  ({100 * n_pass / n_total:.1f}%)')
print(f'Filtered out            : {n_total - n_pass:,}  ({100 * (1 - n_pass / n_total):.1f}%)')
print()
print('Score distribution (full set):')
print(snippets_df['relevance_score'].value_counts().sort_index().to_string())

# ── Save ──────────────────────────────────────────────────────────────────────
qual = (snippets_df[snippets_df['pass_filter']]
        .drop(columns=['pass_filter'])
        .reset_index(drop=True))

OUT_QUAL = '../ln_training/snippets_qual_filter.csv'
qual.to_csv(OUT_QUAL, index=False)
print(f'\nSaved {len(qual):,} filtered snippets → {OUT_QUAL}')
qual.head(3)


Total snippets          : 73,485
Pass (score >= 2)       : 26,238  (35.7%)
Filtered out            : 47,247  (64.3%)

Score distribution (full set):
relevance_score
0     32963
1     14284
2      8950
3      3025
4      6849
5      3608
6      2307
7       435
8       471
9       282
10      203
11       35
12       62
13        3
14        8

Saved 26,238 filtered snippets → ../ln_training/snippets_qual_filter.csv


,article_id,title,outlet,date,snippet,word_count,china,indonesia,nickel,relevance_score
0,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,Mining holds the key to a green future - no wo...,68,0,0,0,2
1,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,That will take a fivefold increase in global r...,65,0,0,0,2
2,urn:contentItem:6313-C6K1-DY4H-K3FJ-00000-00,Mining holds the key to a green future - no wo...,"The Guardian (London), BUSINESS; Version:2, 11...",2021-06-27,Expanding renewable energy is a mineral intens...,55,0,0,1,4


In [40]:
import re
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

CONFLICT_KEYWORDS = [
    'conflict', 'human rights', 'accident', 'pollution', 'protest', 'emission',
    'backlash', 'deforestation', 'displacement', 'contamination', 'exploitation',
    'forced labor', 'forced labour', 'tailings', 'smog', 'health risk',
    'child labor', 'child labour', 'corruption', 'dumping', 'greenwashing',
    'monopoly', 'monopolies', 'trouble', 'abuse', 'bribery', 'confiscation',
    'violation', 'scandal', 'misconduct', 'fraud', 'collapse', 'toxic waste',
    'spill', 'dispute', 'boycott', 'lawsuit', 'litigation', 'repression',
    'unsustainable',
]

conflict_pattern = re.compile(
    r'(?:' + '|'.join(re.escape(kw) + r's?' for kw in CONFLICT_KEYWORDS) + r')',
    flags=re.IGNORECASE
)

# DistilBERT tokenises at ~1.3 tokens/word.
# 100–120 words → ~130–156 tokens, fits comfortably in max_length=192.
TARGET_MIN  = 100
TARGET_MAX  = 120
MIN_SNIPPET =  20
HARD_MAX    = 120


def conflict_centered_snippets(text, pattern=conflict_pattern,
                                target_min=TARGET_MIN, target_max=TARGET_MAX,
                                min_words=MIN_SNIPPET, hard_max=HARD_MAX):
    sentences = sent_tokenize(str(text))
    wc = [len(s.split()) for s in sentences]
    n = len(sentences)

    anchors = [i for i, s in enumerate(sentences) if pattern.search(s)]
    if not anchors:
        return []

    windows = []
    for a in anchors:
        lo, hi = a, a + 1
        total = wc[a]
        while total < target_min:
            added = False
            if hi < n and total + wc[hi] <= target_max:
                total += wc[hi]; hi += 1; added = True
            if lo > 0 and total + wc[lo - 1] <= target_max:
                total += wc[lo - 1]; lo -= 1; added = True
            if not added:
                break
        windows.append((lo, hi))

    windows.sort()
    merged = [list(windows[0])]
    for lo, hi in windows[1:]:
        if lo <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], hi)
        else:
            merged.append([lo, hi])

    result = []
    for lo, hi in merged:
        chunk = ' '.join(sentences[lo:hi])
        words = chunk.split()
        if len(words) > hard_max:
            chunk = ' '.join(words[:hard_max])
        if len(chunk.split()) >= min_words:
            result.append(chunk)
    return result


# ── Apply to all inference articles ──────────────────────────────────────────
rows = []
for _, article in ln_inference.iterrows():
    for snippet in conflict_centered_snippets(article['Text_body']):
        rows.append({
            'article_id' : article['ID'],
            'title'      : article['Title'],
            'outlet'     : article['Outlet_Name'],
            'date'       : article['Date'],
            'snippet'    : snippet,
            'word_count' : len(snippet.split()),
        })

conflict_snippets_df = pd.DataFrame(rows)

text_lower = conflict_snippets_df['snippet'].str.lower()
for kw in ['china', 'indonesia', 'nickel']:
    conflict_snippets_df[kw] = text_lower.str.contains(kw, regex=False).astype(int)

print(f'Articles processed              : {ln_inference["ID"].nunique():,}')
print(f'Articles with >=1 conflict match: {conflict_snippets_df["article_id"].nunique():,}')
print(f'Total conflict-centered snippets: {len(conflict_snippets_df):,}')
print(f'Avg snippets per matched article: {len(conflict_snippets_df) / conflict_snippets_df["article_id"].nunique():.1f}')
print(f'\nWord count distribution:')
print(conflict_snippets_df['word_count'].describe().round(1))


Articles processed              : 1,726
Articles with >=1 conflict match: 1,631
Total conflict-centered snippets: 3,674
Avg snippets per matched article: 2.3

Word count distribution:
count    3674.0
mean      112.1
std         8.6
min        32.0
25%       106.0
50%       114.0
75%       120.0
max       120.0
Name: word_count, dtype: float64


## Conflict-centered snippet extraction

Instead of sliding-window snippets over every article, we locate sentences that mention a conflict keyword and build a **100–120 word window** around each such sentence.
Overlapping windows are merged. This replaces the uniform sliding-window approach for conflict classification —
only text near a conflict mention is passed to the NLI model.

The 100–120 word target is calibrated for the NLI input format: snippet + conflict-type definition together fit comfortably within `max_length=192` tokens without truncation, while providing ~2–3 sentences of context on each side of the keyword mention.

Keywords are matched case-insensitively; `s?` on each term covers regular English plurals.
`monopolies` is listed explicitly because `monopoly → monopolies` is irregular.


In [41]:
import re
import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
from nltk.tokenize import sent_tokenize

CONFLICT_KEYWORDS = [
    'conflict', 'human rights', 'accident', 'pollution', 'protest', 'emission',
    'backlash', 'deforestation', 'displacement', 'contamination', 'exploitation',
    'forced labor', 'forced labour', 'tailings', 'smog', 'health risk',
    'child labor', 'child labour', 'corruption', 'dumping', 'greenwashing',
    'monopoly', 'monopolies', 'trouble', 'abuse', 'bribery', 'confiscation',
    'violation', 'scandal', 'misconduct', 'fraud', 'collapse', 'toxic waste',
    'spill', 'dispute', 'boycott', 'lawsuit', 'litigation', 'repression',
    'unsustainable',
]

conflict_pattern = re.compile(
    r'(?:' + '|'.join(re.escape(kw) + r's?' for kw in CONFLICT_KEYWORDS) + r')',
    flags=re.IGNORECASE
)

# DistilBERT tokenises at ~1.3 tokens/word.
# 100-120 words -> ~130-156 tokens, fits comfortably in max_length=192.
# This gives ~2-3 sentences of context on each side of the keyword mention,
# enough for the model to distinguish between the 5 conflict categories
# (e.g. 'protest + police + repression' vs 'protest + workers + wages').
TARGET_MIN  = 100
TARGET_MAX  = 120
MIN_SNIPPET =  20
HARD_MAX    = 120  # cap merged windows so no snippet exceeds max_length=192 tokens


def conflict_centered_snippets(text, pattern=conflict_pattern,
                                target_min=TARGET_MIN, target_max=TARGET_MAX,
                                min_words=MIN_SNIPPET, hard_max=HARD_MAX):
    sentences = sent_tokenize(str(text))
    wc = [len(s.split()) for s in sentences]
    n = len(sentences)

    anchors = [i for i, s in enumerate(sentences) if pattern.search(s)]
    if not anchors:
        return []

    windows = []
    for a in anchors:
        lo, hi = a, a + 1   # half-open [lo, hi)
        total = wc[a]
        while total < target_min:
            added = False
            if hi < n and total + wc[hi] <= target_max:
                total += wc[hi]; hi += 1; added = True
            if lo > 0 and total + wc[lo - 1] <= target_max:
                total += wc[lo - 1]; lo -= 1; added = True
            if not added:
                break
        windows.append((lo, hi))

    # merge overlapping windows
    windows.sort()
    merged = [list(windows[0])]
    for lo, hi in windows[1:]:
        if lo <= merged[-1][1]:
            merged[-1][1] = max(merged[-1][1], hi)
        else:
            merged.append([lo, hi])

    result = []
    for lo, hi in merged:
        chunk = ' '.join(sentences[lo:hi])
        # hard cap: if a merged window is very long, truncate at word level
        words = chunk.split()
        if len(words) > hard_max:
            chunk = ' '.join(words[:hard_max])
        if len(chunk.split()) >= min_words:
            result.append(chunk)
    return result


# ── apply to all inference articles ─────────────────────────────────────────
rows = []
for _, article in ln_inference.iterrows():
    for snippet in conflict_centered_snippets(article['Text_body']):
        rows.append({
            'article_id' : article['ID'],
            'title'      : article['Title'],
            'outlet'     : article['Outlet_Name'],
            'date'       : article['Date'],
            'snippet'    : snippet,
            'word_count' : len(snippet.split()),
        })

conflict_snippets_df = pd.DataFrame(rows)

# add keyword-presence flags for consistency with snippets_df
text_lower = conflict_snippets_df['snippet'].str.lower()
for kw in ['china', 'indonesia', 'nickel']:
    conflict_snippets_df[kw] = text_lower.str.contains(kw, regex=False).astype(int)

print(f'Articles processed              : {ln_inference["ID"].nunique():,}')
print(f'Articles with >=1 conflict match: {conflict_snippets_df["article_id"].nunique():,}')
print(f'Total conflict-centered snippets: {len(conflict_snippets_df):,}')
print(f'Avg snippets per matched article: {len(conflict_snippets_df) / conflict_snippets_df["article_id"].nunique():.1f}')
print(f'\nWord count distribution:')
print(conflict_snippets_df['word_count'].describe().round(1))


Articles processed              : 1,726
Articles with >=1 conflict match: 1,631
Total conflict-centered snippets: 3,674
Avg snippets per matched article: 2.3

Word count distribution:
count    3674.0
mean      112.1
std         8.6
min        32.0
25%       106.0
50%       114.0
75%       120.0
max       120.0
Name: word_count, dtype: float64


In [42]:
OUT_CONFLICT = r'../ln_training/conflict_snippets_df.xlsx'
conflict_snippets_df.to_excel(OUT_CONFLICT, index=False)
print(f'Saved {len(conflict_snippets_df):,} conflict-centered snippets to: {OUT_CONFLICT}')


Saved 3,674 conflict-centered snippets to: ../ln_training/conflict_snippets_df.xlsx
